# Stage 2 — Four-way RAG Generation

Compare `standard`, `llm_pipeline`, `bert` and `oracle` on the **same questions and retrieved documents**. The three type-conditioned methods share one fixed router and one generator. The standard method uses the same base prompt without conflict-type context.

**Data:** frozen 621 examples (481 train / 68 validation / 72 test). Development and model selection use Validation; Test must not be used to tune the classifier, prompts or evaluation protocol.

**Controls:** matched classifier source fields (`question`, `title`, `short_text`); one generator model; temperature 0; 500 output-token cap; identical generation requests share their saved answer. The prompts are adapted from the [CONFLICTS study](https://arxiv.org/html/2506.08500), not a verbatim reproduction.


## 1. Setup

Run in VS Code or Kaggle with the frozen dataset, split manifest and compatible BERT checkpoint accessible. Install `openai`, `torch`, `transformers`, `pandas`, `numpy` and `scikit-learn`. No external Python helpers are required. Keep Stage 2 and Stage 3 output directories consistent.


In [1]:
# Optional, run once in your environment: %pip install -q openai transformers torch pandas scikit-learn
import os
import json
import time
import random
import getpass
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from sklearn.metrics import accuracy_score, f1_score
from openai import OpenAI
from IPython.display import display


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Experiment configuration

`RUN_MODE` selects `validation_smoke`, `validation_full`, or `final_test`. The current configuration uses `final_test` with `RUN_API=True`.

The spread-only BERT checkpoint is selected using Validation four-class Macro-F1 across seeds 42, 43, and 44. Final Test requires the selected checkpoint and the two confirmation flags.


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

IS_KAGGLE = Path("/kaggle/input").is_dir()
OUTPUT_DIR = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Use files in the current working directory only
DATA_DIR = Path.cwd()

if not (DATA_DIR / "conflicts_621.jsonl").is_file():
    raise FileNotFoundError("Missing conflicts_621.jsonl")

if not (DATA_DIR / "conflicts_original_eval_split.json").is_file():
    raise FileNotFoundError("Missing split manifest")

DATA_PATH = DATA_DIR / "conflicts_621.jsonl"
SPLIT_PATH = DATA_DIR / "conflicts_original_eval_split.json"
ORIGINAL_DATA_PATH = next((p for p in (
    DATA_DIR / "conflicts_458.jsonl", DATA_DIR / "conflicts.jsonl"
) if p.is_file()), None)
if ORIGINAL_DATA_PATH is None:
    raise FileNotFoundError(
        "To supply real source dates for Test generation, place conflicts_458.jsonl "
        "(or conflicts.jsonl) beside conflicts_621.jsonl and the split manifest."
    )

# Validation scores and checkpoints for the three spread-only BERT seeds.
BERT_SEED_CANDIDATES = [
    {
        "seed": 42,
        "best_val_selection_f1_4": 0.6319838057,
        "checkpoint_path": "best_interaction_0_spread_1_original_eval_seed42.pt",
    },
    {
        "seed": 43,
        "best_val_selection_f1_4": 0.6651714514,
        "checkpoint_path": "best_interaction_0_spread_1_original_eval_seed43.pt",
    },
    {
        "seed": 44,
        "best_val_selection_f1_4": 0.6147357310,
        "checkpoint_path": "best_interaction_0_spread_1_original_eval_seed44.pt",
    },
]
EXPECTED_NUM_SEEDS = 3

# Fallback checkpoint for validation runs without seed candidates.
PROVISIONAL_CHECKPOINT_FILENAME = "best_interaction_0_spread_1_original_eval_seed43.pt"
BERT_CHECKPOINT_PATH = ""  # Optional explicit path for a Validation-only checkpoint.

BERT_MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 512
GENERATION_MODEL = "gpt-4o-2024-08-06"
LLM_CLASSIFIER_MODEL = "gpt-4o-2024-08-06"  # Prompted LLM; does NOT use gold labels.
TEMPERATURE = 0
MAX_OUTPUT_TOKENS = 500

DATASET_VERSION = "conflicts_621_frozen"
CLASSIFIER_VERSION = "spread_only_bert"
ROUTER_VERSION = "taxonomy_type_conditioned"
PROMPT_VERSION = "rag_with_source_dates_and_sentence_citations"
LLM_CLASSIFIER_PROMPT_VERSION = "question_title_short_text"

RUN_MODE = "final_test"  # Test only; do not tune based on Test results.
VALIDATION_SMOKE_PER_CLASS = 1  # Increase for broader validation checks (up to 68 total).
RUN_API = True  # Set True after confirming Test settings; API calls incur costs.
ROUTER_AND_PROMPT_FROZEN = True
FINAL_CHECKPOINT_CONFIRMED = True

# Test and Validation results are written to separate files.
VALIDATION_RESULTS_PATH = OUTPUT_DIR / "rag_validation_smoke_fourway.jsonl"
VALIDATION_LLM_LABELS_PATH = OUTPUT_DIR / "rag_llm_labels_validation_smoke.jsonl"
VALIDATION_FULL_RESULTS_PATH = OUTPUT_DIR / "rag_validation_fourway.jsonl"
VALIDATION_FULL_LLM_LABELS_PATH = OUTPUT_DIR / "rag_llm_labels_validation.jsonl"
FINAL_TEST_RESULTS_PATH = OUTPUT_DIR / "rag_test_fourway.jsonl"
FINAL_LLM_LABELS_PATH = OUTPUT_DIR / "rag_llm_labels_test.jsonl"

device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else "cpu"
)
print("Data:", DATA_DIR)
print("BERT inference device:", device)
print("Run mode:", RUN_MODE, "| API enabled:", RUN_API)
print("Max output tokens:", MAX_OUTPUT_TOKENS)


Data: /Users/GXM/03-trimester 2/AIML339/Assignment/7/rag-spread 新数据
BERT inference device: mps
Run mode: final_test | API enabled: True
Max output tokens: 500


## 3. Load data and fixed splits

Validate the dataset SHA-256 against the split manifest. Only the 68 Validation examples are used for development; the 72 original Test examples remain held out until the protocol is frozen.


In [3]:
LABELS = [
    "No conflict",
    "Complementary information",
    "Conflicting opinions and research outcomes",
    "Conflict due to outdated information",
    "Conflict due to misinformation",
]
label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}

def load_jsonl(path):
    if not Path(path).is_file():
        raise FileNotFoundError(path)
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

data = load_jsonl(DATA_PATH)
assert len(data) == 621
rows = []
for example_id, item in enumerate(data):
    assert item["conflict_type"] in label2id
    rows.append({
        "example_id": example_id,
        "question": item["question"],
        "documents": item["search_results"],
        "gold_label": item["conflict_type"],
        "label": label2id[item["conflict_type"]],
    })

# Generator-only metadata: restore genuine source dates by original example ID
# and document position. BERT and LLM classifiers continue using ONLY
# question + title + short_text; no date enters either classifier.
original_records = load_jsonl(ORIGINAL_DATA_PATH)
if len(original_records) != 458:
    raise ValueError("Original CONFLICTS file must contain exactly 458 rows")
ORIGINAL_DATA_SHA256 = file_hash_original = hashlib.sha256(ORIGINAL_DATA_PATH.read_bytes()).hexdigest()

def _usable_source_date(raw):
    if raw is None:
        return None
    value = str(raw).strip()
    return None if value.lower() in {"", "na", "n/a", "nan", "none", "null"} else value

def _generation_documents(example_id, prepared_documents):
    if int(example_id) >= 458:
        return [dict(d) for d in prepared_documents]  # synthetic train-only rows
    original = original_records[int(example_id)]
    if len(original["search_results"]) != len(prepared_documents):
        raise ValueError(f"Document count changed at example_id={example_id}")
    restored = []
    for number, (source_doc, prepared_doc) in enumerate(
        zip(original["search_results"], prepared_documents), start=1
    ):
        for key in ("title", "short_text"):
            if str(source_doc.get(key) or "").strip() != str(prepared_doc.get(key) or "").strip():
                raise ValueError(f"Document alignment mismatch: example_id={example_id}, doc={number}, key={key}")
        item = dict(prepared_doc)
        date = _usable_source_date(source_doc.get("date"))
        if date:
            item["date"] = date
        restored.append(item)
    return restored

for record in rows:
    i = int(record["example_id"])
    if i < 458:
        assert original_records[i]["question"] == record["question"]
        assert original_records[i]["conflict_type"] == record["gold_label"]
    record["generation_documents"] = _generation_documents(i, record["documents"])
df = pd.DataFrame(rows)
print("Verified original source and restored date metadata for generation only")
print("Loaded examples:", len(df))
print(df.gold_label.value_counts())

Verified original source and restored date metadata for generation only
Loaded examples: 621
gold_label
No conflict                                   161
Complementary information                     115
Conflicting opinions and research outcomes    115
Conflict due to misinformation                115
Conflict due to outdated information          115
Name: count, dtype: int64


In [4]:
with open(SPLIT_PATH, encoding="utf-8") as f:
    split_manifest = json.load(f)
canonical = "\n".join(json.dumps(item, ensure_ascii=False, sort_keys=True) for item in data)
dataset_sha256 = hashlib.sha256(canonical.encode("utf-8")).hexdigest()
assert dataset_sha256 == split_manifest["dataset_sha256"], "Dataset and split manifest differ"

train_indices = split_manifest["train_indices"]
validation_indices = split_manifest["validation_indices"]
test_indices = split_manifest["test_indices"]
assert (len(train_indices), len(validation_indices), len(test_indices)) == (481, 68, 72)
assert len(set(train_indices + validation_indices + test_indices)) == 621
assert set(train_indices + validation_indices + test_indices) == set(range(621))
assert all(i < 458 for i in validation_indices + test_indices)
assert all(i >= 458 for i in split_manifest["synthetic_train_indices"])

train_df = df.iloc[train_indices].copy().reset_index(drop=True)
val_df = df.iloc[validation_indices].copy().reset_index(drop=True)
test_df = df.iloc[test_indices].copy().reset_index(drop=True)
print("Frozen split:", len(train_df), len(val_df), len(test_df))
print("Test class support:", test_df.gold_label.value_counts().to_dict())

Frozen split: 481 68 72
Test class support: {'No conflict': 25, 'Complementary information': 18, 'Conflicting opinions and research outcomes': 18, 'Conflict due to outdated information': 10, 'Conflict due to misinformation': 1}


## 4. Load Spread-only BERT (selected using Validation only)


In [5]:
tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_NAME)


# Classifier inputs use only title and short_text in addition to the question.
def build_document_text(doc):
    title = doc.get("title", "") or ""
    short_text = doc.get("short_text", "") or ""
    return "Title: " + title + (". Text: " + short_text if short_text else "")


class ConflictDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.dataframe = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        all_input_ids = []
        all_attention_masks = []

        for doc in row["documents"]:
            text = build_document_text(doc)

            encoding = self.tokenizer(
                row["question"],
                text,
                padding="max_length",
                truncation="only_second",
                max_length=self.max_length,
                return_tensors="pt",
            )

            all_input_ids.append(
                encoding["input_ids"].squeeze(0)
            )

            all_attention_masks.append(
                encoding["attention_mask"].squeeze(0)
            )

        return {
            "input_ids": torch.stack(all_input_ids),
            "attention_mask": torch.stack(all_attention_masks),
            "label": torch.tensor(
                int(row["label"]),
                dtype=torch.long,
            ),
        }


def collate_fn(batch):
    batch_size = len(batch)
    max_docs = max(
        item["input_ids"].size(0)
        for item in batch
    )

    seq_length = batch[0]["input_ids"].size(1)

    batch_input_ids = torch.full(
        (batch_size, max_docs, seq_length),
        tokenizer.pad_token_id,
        dtype=torch.long,
    )

    batch_attention_masks = torch.zeros(
        (batch_size, max_docs, seq_length),
        dtype=torch.long,
    )

    document_mask = torch.zeros(
        (batch_size, max_docs),
        dtype=torch.float,
    )

    labels = []

    for i, item in enumerate(batch):
        num_docs = item["input_ids"].size(0)

        batch_input_ids[i, :num_docs] = item["input_ids"]
        batch_attention_masks[i, :num_docs] = item["attention_mask"]
        document_mask[i, :num_docs] = 1

        labels.append(item["label"])

    return {
        "input_ids": batch_input_ids,
        "attention_mask": batch_attention_masks,
        "document_mask": document_mask,
        "label": torch.stack(labels),
    }


class InteractionAwareBERT(torch.nn.Module):
    def __init__(
        self,
        model_name,
        num_labels=5,
        use_interaction=True,
        use_spread=True,
        num_heads=8,
        num_transformer_layers=1,
        dropout=0.1,
    ):
        super().__init__()

        self.bert = BertModel.from_pretrained(model_name)
        self.hidden_size = self.bert.config.hidden_size

        self.use_interaction = use_interaction
        self.use_spread = use_spread

        if use_interaction:
            encoder_layer = torch.nn.TransformerEncoderLayer(
                d_model=self.hidden_size,
                nhead=num_heads,
                dim_feedforward=self.hidden_size * 4,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
            )

            self.document_encoder = torch.nn.TransformerEncoder(
                encoder_layer,
                num_layers=num_transformer_layers,
                enable_nested_tensor=False,
            )

        classifier_input_size = (
            self.hidden_size * 2
            if use_spread
            else self.hidden_size
        )

        self.dropout = torch.nn.Dropout(dropout)

        self.classifier = torch.nn.Linear(
            classifier_input_size,
            num_labels,
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        document_mask,
    ):
        batch_size, num_docs, seq_length = input_ids.shape

        flat_input_ids = input_ids.reshape(
            batch_size * num_docs,
            seq_length,
        )

        flat_attention_mask = attention_mask.reshape(
            batch_size * num_docs,
            seq_length,
        )

        outputs = self.bert(
            input_ids=flat_input_ids,
            attention_mask=flat_attention_mask,
        )

        document_embeddings = outputs.last_hidden_state[:, 0, :]

        document_embeddings = document_embeddings.reshape(
            batch_size,
            num_docs,
            self.hidden_size,
        )

        if self.use_interaction:
            padding_mask = document_mask == 0

            document_embeddings = self.document_encoder(
                document_embeddings,
                src_key_padding_mask=padding_mask,
            )

        mask = document_mask.unsqueeze(-1)

        summed = (
            document_embeddings * mask
        ).sum(dim=1)

        counts = (
            document_mask
            .sum(dim=1, keepdim=True)
            .clamp(min=1)
        )

        mean_embedding = summed / counts

        if self.use_spread:
            valid_mask = (
                document_mask
                .unsqueeze(-1)
                .bool()
            )

            max_embedding = (
                document_embeddings
                .masked_fill(
                    ~valid_mask,
                    float("-inf"),
                )
                .max(dim=1)
                .values
            )

            min_embedding = (
                document_embeddings
                .masked_fill(
                    ~valid_mask,
                    float("inf"),
                )
                .min(dim=1)
                .values
            )

            spread_embedding = (
                max_embedding
                - min_embedding
            )

            combined = torch.cat(
                [
                    mean_embedding,
                    spread_embedding,
                ],
                dim=1,
            )

        else:
            combined = mean_embedding

        combined = self.dropout(combined)

        return self.classifier(combined)


In [6]:
# Select by Validation 4-class Macro-F1; never by Test or RAG results.
selection_sha256 = None
selected_seed = None
selected_val_f1_4 = None
selection_is_provisional = False

def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024*1024), b""):
            digest.update(block)
    return digest.hexdigest()

if BERT_SEED_CANDIDATES:
    selection_df = pd.DataFrame(BERT_SEED_CANDIDATES)
    must_have = {"seed", "best_val_selection_f1_4", "checkpoint_path"}
    if not must_have.issubset(selection_df.columns):
        raise ValueError(f"Add exactly these fields for each seed: {sorted(must_have)}")
    if (len(selection_df) != EXPECTED_NUM_SEEDS or
            selection_df.seed.isna().any() or selection_df.seed.astype(str).duplicated().any()):
        raise ValueError(f"Enter exactly {EXPECTED_NUM_SEEDS} distinct BERT seeds")
    selection_df["best_val_selection_f1_4"] = pd.to_numeric(
        selection_df.best_val_selection_f1_4, errors="raise")
    if (not np.isfinite(selection_df.best_val_selection_f1_4).all() or
            not selection_df.best_val_selection_f1_4.between(0, 1).all()):
        raise ValueError("Use valid Validation 4-class Macro-F1 scores in [0,1]")
    if (selection_df.checkpoint_path.isna().any() or
            selection_df.checkpoint_path.astype(str).str.strip().eq("").any()):
        raise ValueError("Every seed needs a checkpoint_path")
    selection_df["seed"] = selection_df["seed"].astype(str)
    selection_df = selection_df.sort_values(
        ["best_val_selection_f1_4", "seed"], ascending=[False, True]).reset_index(drop=True)
    best = selection_df.iloc[0]
    selected_seed = str(best.seed)
    selected_val_f1_4 = float(best.best_val_selection_f1_4)
    raw_path = Path(str(best.checkpoint_path)).expanduser()
    checkpoint_path = raw_path if raw_path.is_absolute() else DATA_DIR / raw_path
    selection_sha256 = hashlib.sha256(json.dumps(
        BERT_SEED_CANDIDATES, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
    display(selection_df[["seed", "best_val_selection_f1_4", "checkpoint_path"]])
    print("Selected BERT seed from Validation only:", selected_seed, selected_val_f1_4)
else:
    if RUN_MODE == "final_test":
        raise RuntimeError(
            "Fill BERT_SEED_CANDIDATES in the settings cell with 3 REAL Validation "
            "Macro-F1 scores and checkpoint paths before validation_full/final_test. "
            "Do not use Test or RAG scores to select a model.")
    selection_is_provisional = True
    if BERT_CHECKPOINT_PATH:
        checkpoint_path = Path(BERT_CHECKPOINT_PATH).expanduser()
    else:
        matches = sorted(set(DATA_DIR.rglob(PROVISIONAL_CHECKPOINT_FILENAME)))
        if len(matches) != 1:
            raise FileNotFoundError(
                "Provisional Validation checkpoint not found. Set BERT_CHECKPOINT_PATH or fill "
                "BERT_SEED_CANDIDATES in the settings cell.")
        checkpoint_path = matches[0]
    print("WARNING: provisional checkpoint; Validation only, not approved for final Test")

if not checkpoint_path.is_file():
    raise FileNotFoundError(checkpoint_path)
if RUN_MODE == "final_test" and (selection_is_provisional or not FINAL_CHECKPOINT_CONFIRMED):
    raise RuntimeError("Final Test requires validation-selected seed and explicit confirmation")
if BERT_CHECKPOINT_PATH and BERT_SEED_CANDIDATES:
    raise ValueError("With BERT_SEED_CANDIDATES filled, do not override BERT_CHECKPOINT_PATH")
checkpoint_sha256 = file_sha256(checkpoint_path)
model = InteractionAwareBERT(
    BERT_MODEL_NAME, num_labels=5, use_interaction=False, use_spread=True)
weights = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
model.load_state_dict(weights, strict=True)
model = model.to(device).eval()
print("Checkpoint:", checkpoint_path)
print("Checkpoint SHA-256:", checkpoint_sha256[:16])


,seed,best_val_selection_f1_4,checkpoint_path
0,43,0.665171,best_interaction_0_spread_1_original_eval_seed...
1,42,0.631984,best_interaction_0_spread_1_original_eval_seed...
2,44,0.614736,best_interaction_0_spread_1_original_eval_seed...


Selected BERT seed from Validation only: 43 0.6651714514


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7982.66it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Checkpoint: /Users/GXM/03-trimester 2/AIML339/Assignment/7/rag-spread 新数据/best_interaction_0_spread_1_original_eval_seed43.pt
Checkpoint SHA-256: 46007fb18a1c8065


## 5. Predict conflict labels

BERT predictions are made before generation. On Test, the classifier weights and routing rules must already be frozen. Gold labels are used only for the Oracle condition and evaluation.


In [7]:
def predict_dataframe(dataframe):
    dataset = ConflictDataset(
        dataframe,
        tokenizer,
        MAX_LENGTH,
    )

    loader = DataLoader(
        dataset,
        batch_size=1,
        shuffle=False,
        collate_fn=collate_fn,
    )

    predicted_ids = []
    true_ids = []

    model.eval()

    with torch.no_grad():
        for batch in loader:
            logits = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
                document_mask=batch["document_mask"].to(device),
            )

            predicted_ids.append(
                int(
                    torch.argmax(
                        logits,
                        dim=1,
                    ).item()
                )
            )

            true_ids.append(
                int(batch["label"].item())
            )

    output_df = dataframe.copy()

    output_df["predicted_label"] = [
        id2label[i]
        for i in predicted_ids
    ]

    accuracy = accuracy_score(
        true_ids,
        predicted_ids,
    )

    macro_f1 = f1_score(
        true_ids, predicted_ids, labels=list(range(5)),
        average="macro", zero_division=0,
    )
    macro_f1_4 = f1_score(
        true_ids, predicted_ids, labels=list(range(4)),
        average="macro", zero_division=0,
    )
    print("Four-class selection F1 (excl. misinformation):", round(macro_f1_4, 4))

    return output_df, accuracy, macro_f1


In [8]:
if RUN_MODE in ("validation_smoke", "validation_full"):
    print("Predicting validation examples...")
    val_pred_df, val_accuracy, val_macro_f1 = predict_dataframe(val_df)
    print("Validation Accuracy:", round(val_accuracy, 4))
    print("Validation 5-class Macro-F1:", round(val_macro_f1, 4))
elif RUN_MODE == "final_test":
    if not ROUTER_AND_PROMPT_FROZEN or not FINAL_CHECKPOINT_CONFIRMED:
        raise RuntimeError(
            "Final test requires frozen prompts and a checkpoint trained on the "
            "481/68/72 split. Set both confirmation flags after verification."
        )
    print("Predicting held-out test examples (without using labels for tuning)...")
    test_pred_df, test_accuracy, test_macro_f1 = predict_dataframe(test_df)
    print("Test Accuracy (descriptive only):", round(test_accuracy, 4))
    print("Test 5-class Macro-F1 (descriptive only):", round(test_macro_f1, 4))
else:
    raise ValueError("RUN_MODE must be 'validation_smoke', 'validation_full' or 'final_test'")

Predicting held-out test examples (without using labels for tuning)...
Four-class selection F1 (excl. misinformation): 0.6157
Test Accuracy (descriptive only): 0.6389
Test 5-class Macro-F1 (descriptive only): 0.4925


## 6. Test protocol boundary

The 72 original Test examples are not used for prompt design, checkpoint selection or Judge calibration.


In [9]:
if RUN_MODE == "final_test":
    final_test_df = test_pred_df.copy().reset_index(drop=True)
    assert len(final_test_df) == 72
    print("Downstream test examples:", len(final_test_df))
else:
    print("Validation mode: held-out test examples are not used.")


Downstream test examples: 72


## 7. Fixed taxonomy and routing

LLM-guided, BERT-guided and Oracle RAG receive the same type-conditioned instruction, selected by different labels. There is no browsing, reranking or document filtering.


In [ ]:
CONFLICT_TAXONOMY = {
    "No conflict": {
        "definition": "Retrieved sources offer equivalent or nearly equivalent answers. Minor phrasing or detail differences do not constitute a disagreement; irrelevant passages are not a conflict.",
        "expected_behavior": "Give a clear and direct unified answer, without inventing alternative viewpoints or unnecessary uncertainty."
    },
    "Complementary information": {
        "definition": "Sources provide different but mutually compatible answers, facets, or interpretations of an underspecified question.",
        "expected_behavior": "Consolidate the relevant compatible answers and distinct facets into one coherent response; do not frame them as a debate."
    },
    "Conflicting opinions and research outcomes": {
        "definition": "Sources substantively disagree because of differing opinions, contested interpretations, or incompatible research findings.",
        "expected_behavior": "Explicitly acknowledge the substantive debate and neutrally summarize the competing, evidence-backed viewpoints without forcing agreement."
    },
    "Conflict due to outdated information": {
        "definition": "Different answers arise because facts changed over time and older sources no longer reflect the latest state.",
        "expected_behavior": "Prioritize the latest information supported by the retrieved documents; optionally identify older figures as historical rather than equally current alternatives."
    },
    "Conflict due to misinformation": {
        "definition": "Sources make incompatible factual claims and at least one claim is misleading, false, or inaccurate.",
        "expected_behavior": "Disregard inaccurate claims when the evidence establishes they are wrong; give the supported, reliable answer. If the retrieved evidence cannot identify the accurate claim, express that uncertainty instead of guessing."
    }
}
assert set(CONFLICT_TAXONOMY) == set(LABELS)

STANDARD_INSTRUCTION = "Answer the question using the supplied search results."

def route_instruction(label):
    if label not in CONFLICT_TAXONOMY:
        raise ValueError(f"Unknown conflict label: {label}")
    item = CONFLICT_TAXONOMY[label]
    return (f"Conflict type: {label}\n"
            f"Definition: {item['definition']}\n"
            f"Expected response behavior: {item['expected_behavior']}")


# Classifier examples are synthetic and are not drawn from the evaluation questions.
CLASSIFICATION_EXAMPLES = {
    "No conflict": ["Q: What is the museum's closing time? Two pages both say 18:00. -> No conflict"],
    "Complementary information": ["Q: What does a local service offer? One page lists lessons and another lists equipment loans; both apply. -> Complementary information"],
    "Conflicting opinions and research outcomes": ["Q: Is a disputed workplace policy effective? Credible reports give opposing results. -> Conflicting opinions and research outcomes"],
    "Conflict due to outdated information": ["Q: Who leads an organization now? An older page names the former manager; a newly dated announcement names the replacement. -> Conflict due to outdated information"],
    "Conflict due to misinformation": ["Q: What year was a bridge completed? Two independent records support 1998, while an unrelated page mistakenly claims 1988. -> Conflict due to misinformation"],
}
assert set(CLASSIFICATION_EXAMPLES) == set(LABELS)

LLM_CLASSIFIER_INSTRUCTION = """Classify the relationship among the retrieved search results WITH RESPECT TO
THE USER'S QUESTION. Treat the five categories as mutually exclusive and choose
exactly one. Consider which documents actually answer the question; ignore
irrelevant pages. Distinguish complementary partial answers from incompatible
opinions, older facts from current facts, and demonstrably wrong factual claims
from merely different dates/events. Do NOT use a reference answer or any gold label.
Return a single valid JSON object with one key: {"label": "<exact category name>"}.
No answer to the question, additional keys, prose or markdown.
"""

def format_classifier_documents(documents):
    """Format retrieved documents using the same text fields as BERT."""
    return "\n\n".join(
        f"Document {i}\n{build_document_text(doc)}"
        for i, doc in enumerate(documents, 1)
    )


def build_llm_classification_prompt(row):
    sections = [LLM_CLASSIFIER_INSTRUCTION.strip(), "Five category definitions and illustrative examples:"]
    for label in LABELS:
        info = CONFLICT_TAXONOMY[label]
        examples = "\n".join("  - " + x for x in CLASSIFICATION_EXAMPLES[label])
        sections.append(f"Category: {label}\nDefinition: {info['definition']}\nExamples:\n{examples}")
    sections.extend([
        "User question:\n" + row["question"],
        "Retrieved search results (matched BERT information fields):\n" + format_classifier_documents(row["documents"]),
        "Select exactly one label from: " + "; ".join(LABELS),
    ])
    return "\n\n".join(sections)


PROMPT_TEXT_SHA256 = hashlib.sha256((LLM_CLASSIFIER_INSTRUCTION + json.dumps(CONFLICT_TAXONOMY, sort_keys=True) + json.dumps(CLASSIFICATION_EXAMPLES, sort_keys=True)).encode()).hexdigest()


def classify_row(client, row, model="gpt-4o-2024-08-06", max_retries=3):
    """Predict a conflict label using the fixed classification prompt."""
    import time
    prompt = build_llm_classification_prompt(row)
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model, temperature=0,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": "Return a JSON object with exactly one of the five category labels."},
                    {"role": "user", "content": prompt},
                ],
            )
            raw = response.choices[0].message.content or ""
            obj = json.loads(raw)
            if not isinstance(obj, dict) or set(obj) != {"label"} or obj["label"] not in LABELS:
                raise ValueError(f"Invalid LLM classification: {raw[:300]}")
            usage = response.usage
            return {
                "llm_predicted_label": obj["label"],
                "classifier_input_tokens": getattr(usage, "prompt_tokens", None),
                "classifier_output_tokens": getattr(usage, "completion_tokens", None),
            }
        except Exception:
            if attempt == max_retries - 1:
                raise
            time.sleep(2 ** attempt)



FOUR_LABELS = [s for s in LABELS if s != "Conflict due to misinformation"]

def classification_metrics(y_true, y_pred):
    y_true, y_pred = list(y_true), list(y_pred)
    if not set(y_true + y_pred).issubset(LABELS):
        raise ValueError("Labels must use the exact five canonical class names")
    if len(y_true) != len(y_pred) or not y_true:
        raise ValueError("Inputs must have equal nonzero lengths")
    return {
        "n": len(y_true),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1_5class": float(f1_score(y_true, y_pred, labels=LABELS,
                                           average="macro", zero_division=0)),
        "macro_f1_4class": float(f1_score(y_true, y_pred, labels=FOUR_LABELS,
                                           average="macro", zero_division=0)),
    }


assert LABELS == list(label2id.keys()), "Canonical label order mismatch"
router_review = pd.DataFrame([
    {"label": label, "definition": spec["definition"],
     "expected_behavior": spec["expected_behavior"]}
    for label, spec in CONFLICT_TAXONOMY.items()
])
display(router_review)
print("Classifier prompt SHA-256:", PROMPT_TEXT_SHA256[:16])


,label,definition,expected_behavior
0,No conflict,Retrieved sources offer equivalent or nearly e...,"Give a clear and direct unified answer, withou..."
1,Complementary information,Sources provide different but mutually compati...,Consolidate the relevant compatible answers an...
2,Conflicting opinions and research outcomes,Sources substantively disagree because of diff...,Explicitly acknowledge the substantive debate ...
3,Conflict due to outdated information,Different answers arise because facts changed ...,Prioritize the latest information supported by...
4,Conflict due to misinformation,Sources make incompatible factual claims and a...,Disregard inaccurate claims when the evidence ...


Classifier prompt SHA-256: 9b91a912cda2e1d9


## 8. Generation prompt

All four methods receive the same question and retrieved evidence (`title`, `date` when available, and `short_text`). The shared prompt requests document-number citations, while conflict-type instructions are added for the three guided methods.

In [11]:
def format_documents(documents):
    parts = []
    for i, doc in enumerate(documents, start=1):
        lines = [f"[Document {i}]"]
        for name, key in (("Title", "title"), ("Source date", "date"),
                          ("Relevant passage", "short_text")):
            value = str(doc.get(key) or "").strip()
            if value:
                lines.append(f"{name}: {value}")
        parts.append("\n".join(lines))
    return "\n\n".join(parts)

# All methods receive identical wording here; only typed context differs.
SHARED_PROMPT_PREFIX = """Answer the following question based on the retrieved search results.
Cite the relevant result number (for example [1]) for each factual sentence.
Provide a well-supported answer to the question. Do not invent unsupported facts.

Question:
{question}

Search results:
{documents}
"""

def build_rag_prompt(row, condition):
    question = row["question"]
    documents = format_documents(row["generation_documents"])
    if condition == "standard":
        routed_label = None
        strategy = STANDARD_INSTRUCTION
        type_context = ""
    elif condition in ("llm_pipeline", "bert", "oracle"):
        if condition == "llm_pipeline":
            routed_label = row.get("llm_predicted_label")
            if not isinstance(routed_label, str) or routed_label not in LABELS:
                raise ValueError("LLM label not saved for this example; run LLM classification first")
        else:
            routed_label = row["predicted_label"] if condition == "bert" else row["gold_label"]
        strategy = route_instruction(routed_label)
        type_context = "\nAdditional conflict-type context:\n" + strategy + "\n"
    else:
        raise ValueError("Unknown condition; expected standard/llm_pipeline/bert/oracle")
    # All conditions use this same prefix, evidence and closing instruction.
    prompt = (SHARED_PROMPT_PREFIX.format(question=question, documents=documents)
              + type_context + "\nAnswer with source citations: ")
    return prompt, routed_label, strategy


## 9. Select evaluation split

Validation smoke is a limited dry run, full Validation covers 68 questions, and final Test covers 72 questions. Each question has four generation conditions.


In [12]:
if RUN_MODE == "validation_smoke":
    run_df = (val_pred_df.groupby("gold_label", group_keys=False)
              .head(VALIDATION_SMOKE_PER_CLASS).reset_index(drop=True))
    RESULTS_PATH = VALIDATION_RESULTS_PATH
    LLM_LABELS_PATH = VALIDATION_LLM_LABELS_PATH
elif RUN_MODE == "validation_full":
    run_df = val_pred_df.copy().reset_index(drop=True)
    RESULTS_PATH = VALIDATION_FULL_RESULTS_PATH
    LLM_LABELS_PATH = VALIDATION_FULL_LLM_LABELS_PATH
elif RUN_MODE == "final_test":
    assert ROUTER_AND_PROMPT_FROZEN and FINAL_CHECKPOINT_CONFIRMED
    run_df = final_test_df.copy().reset_index(drop=True)
    RESULTS_PATH = FINAL_TEST_RESULTS_PATH
    LLM_LABELS_PATH = FINAL_LLM_LABELS_PATH
else:
    raise ValueError("Unknown RUN_MODE")

print("Run mode:", RUN_MODE)
print("Examples:", len(run_df), "| Answers expected:", len(run_df) * 4)
print("LLM labels file:", LLM_LABELS_PATH)
print("Results path:", RESULTS_PATH)
display(run_df[["example_id", "question", "gold_label", "predicted_label"]].head(10))

Run mode: final_test
Examples: 72 | Answers expected: 288
LLM labels file: /Users/GXM/03-trimester 2/AIML339/Assignment/7/rag-spread 新数据/rag_llm_labels_test.jsonl
Results path: /Users/GXM/03-trimester 2/AIML339/Assignment/7/rag-spread 新数据/rag_test_fourway.jsonl


,example_id,question,gold_label,predicted_label
0,4,when was coffee first made into a drink?,Complementary information,Complementary information
1,7,Where did the expression a bridge too far come...,Conflict due to misinformation,No conflict
2,12,When does this year's Passover start?,Conflict due to outdated information,Conflict due to outdated information
3,19,when was the last person executed,Complementary information,Complementary information
4,39,How many times csk reached final in ipl?,No conflict,No conflict
5,44,How many cities in india has metro train?,Complementary information,Conflict due to outdated information
6,45,cost of plumber,Complementary information,Complementary information
7,46,What is the title of Stephen King's most recen...,No conflict,Conflict due to outdated information
8,49,When did the british come to new zealand?,Complementary information,Complementary information
9,56,Who won the latest Ballon D'or?,Conflict due to outdated information,Conflict due to outdated information


## 10. Inspect classifier and generation prompts

Check that the prompted LLM classifier uses only `question`, `title` and `short_text`. Gold labels must not appear in classifier inputs.


In [13]:
preview_row = run_df.iloc[0]
print("\n=== Prompted LLM classifier preview (no gold label) ===\n")
print(build_llm_classification_prompt(preview_row)[:5000])
for condition in ["standard", "bert", "oracle"]:
    prompt, routed_label, strategy = build_rag_prompt(preview_row, condition)
    print("\n" + "=" * 70)
    print("CONDITION:", condition, "| ROUTED LABEL:", routed_label)
    print(prompt[:3200])
print("\nLLM Pipeline generator preview follows after LLM labels have been loaded.")



=== Prompted LLM classifier preview (no gold label) ===

Classify the relationship among the retrieved search results WITH RESPECT TO
THE USER'S QUESTION. Treat the five categories as mutually exclusive and choose
exactly one. Consider which documents actually answer the question; ignore
irrelevant pages. Distinguish complementary partial answers from incompatible
opinions, older facts from current facts, and demonstrably wrong factual claims
from merely different dates/events. Do NOT use a reference answer or any gold label.
Return a single valid JSON object with one key: {"label": "<exact category name>"}.
No answer to the question, additional keys, prose or markdown.

Five category definitions and illustrative examples:

Category: No conflict
Definition: Retrieved sources offer equivalent or nearly equivalent answers. Minor phrasing or detail differences do not constitute a disagreement; irrelevant passages are not a conflict.
Examples:
  - Q: What is the museum's closing time? Two

## 11. API access

`RUN_API` controls prompted LLM classification and answer generation. The current configuration enables API calls, while previously completed requests are reused from the saved caches.


In [14]:
client = None
if RUN_API:
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key and IS_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            api_key = UserSecretsClient().get_secret("OPENAI_API_KEY")
        except Exception as exc:
            raise RuntimeError(
                "Set the Kaggle Secret 'OPENAI_API_KEY' and grant notebook access "
                "before running API calls in the background."
            ) from exc
    if not api_key:
        api_key = getpass.getpass("Enter OPENAI_API_KEY: ")
    client = OpenAI(api_key=api_key)
else:
    print("LLM classifier and generator API calls disabled.")

In [15]:
def generate_answer(
    prompt,
    max_retries=3,
):
    if not RUN_API:
        raise RuntimeError(
            "RUN_API is False."
        )

    for attempt in range(max_retries):
        try:
            response = client.responses.create(
                model=GENERATION_MODEL,
                input=prompt,
                temperature=TEMPERATURE,
                max_output_tokens=MAX_OUTPUT_TOKENS,
            )

            usage = getattr(
                response,
                "usage",
                None,
            )

            input_tokens = (
                getattr(
                    usage,
                    "input_tokens",
                    None,
                )
                if usage
                else None
            )

            output_tokens = (
                getattr(
                    usage,
                    "output_tokens",
                    None,
                )
                if usage
                else None
            )

            total_tokens = (
                getattr(
                    usage,
                    "total_tokens",
                    None,
                )
                if usage
                else None
            )

            near_token_limit = (
                output_tokens is not None
                and output_tokens >= 0.90 * MAX_OUTPUT_TOKENS
            )

            answer_text = (response.output_text or "").strip()
            if not answer_text:
                raise RuntimeError("Generator returned an empty answer")
            return {
                "answer": answer_text,
                "input_tokens": input_tokens,
                "output_tokens": output_tokens,
                "total_tokens": total_tokens,
                "near_token_limit": near_token_limit,
            }

        except Exception as e:
            if attempt == max_retries - 1:
                raise

            wait_seconds = 2 ** attempt

            print(
                f"API error: {type(e).__name__}. "
                f"Retrying in {wait_seconds}s..."
            )

            time.sleep(wait_seconds)

## 12. Prompted LLM labels and resumable cache

Predict each question's conflict label once with `question + title + short_text`, then store it in a fingerprinted JSONL. Resume only when the saved classifier configuration matches.


In [16]:
# A cached label is reused only when the full classifier configuration matches.
CONDITIONS = ["standard", "llm_pipeline", "bert", "oracle"]
LLM_CLASSIFIER_CONFIG = {
    "dataset_sha256": dataset_sha256,
    "split_sha256": hashlib.sha256(SPLIT_PATH.read_bytes()).hexdigest(),
    "run_mode": RUN_MODE,
    "example_ids": sorted(int(i) for i in run_df.example_id),
    "model": LLM_CLASSIFIER_MODEL,
    "temperature": 0,
    "prompt_version": LLM_CLASSIFIER_PROMPT_VERSION,
    "instruction": LLM_CLASSIFIER_INSTRUCTION,
    "taxonomy": CONFLICT_TAXONOMY,
    "illustrative_examples": CLASSIFICATION_EXAMPLES,
    "full_formatted_classification_inputs_hash": hashlib.sha256("\n".join(build_llm_classification_prompt(r) for _, r in run_df.iterrows()).encode()).hexdigest(),
    "evidence_fields": ["question", "title", "short_text"],
    "classifier_input_format": "question_title_short_text",
    "shared_prompt_text_sha256": PROMPT_TEXT_SHA256,
}
llm_classifier_config_hash = hashlib.sha256(json.dumps(
    LLM_CLASSIFIER_CONFIG, ensure_ascii=False, sort_keys=True
).encode()).hexdigest()

def classify_with_llm(row, max_retries=3):
    return classify_row(client, row, model=LLM_CLASSIFIER_MODEL, max_retries=max_retries)

# Validate saved predictions before optionally calling the API.
label_records = load_jsonl(LLM_LABELS_PATH) if LLM_LABELS_PATH.exists() else []
if any(r.get("classifier_config_hash") != llm_classifier_config_hash for r in label_records):
    raise ValueError(f"Different LLM classifier experiment in {LLM_LABELS_PATH}; rename that file")
llm_label_map = {}
for r in label_records:
    example_id = int(r["example_id"])
    if example_id in llm_label_map or example_id not in set(run_df.example_id):
        raise ValueError("Duplicate or unexpected LLM-classifier example ID")
    if r.get("llm_predicted_label") not in LABELS:
        raise ValueError("Invalid cached LLM predicted label")
    llm_label_map[example_id] = r["llm_predicted_label"]

if RUN_API:
    for _, row in run_df.iterrows():
        example_id = int(row["example_id"])
        if example_id in llm_label_map:
            continue
        cls = classify_with_llm(row)
        item = {
            "example_id": example_id,
            "question": row["question"],
            "llm_predicted_label": cls["llm_predicted_label"],
            "classifier_model": LLM_CLASSIFIER_MODEL,
            "classifier_prompt_version": LLM_CLASSIFIER_PROMPT_VERSION,
            "classifier_config_hash": llm_classifier_config_hash,
            "classifier_input_tokens": cls["classifier_input_tokens"],
            "classifier_output_tokens": cls["classifier_output_tokens"],
        }
        with open(LLM_LABELS_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
        llm_label_map[example_id] = cls["llm_predicted_label"]
        print("Saved LLM classifier:", len(llm_label_map), "/", len(run_df), flush=True)
else:
    print(f"LLM predictions saved: {len(llm_label_map)} / {len(run_df)}; zero API calls.")

run_df = run_df.copy()
run_df["llm_predicted_label"] = run_df.example_id.map(llm_label_map)
if len(llm_label_map) == len(run_df):
    print("\nClassifier comparison (descriptive; don't use Test to tune):")
    display(run_df[["example_id", "gold_label", "predicted_label", "llm_predicted_label"]])
    p, label, strategy = build_rag_prompt(run_df.iloc[0], "llm_pipeline")
    print("\nLLM Pipeline generator preview:", label, "\n", p[:2500])



Classifier comparison (descriptive; don't use Test to tune):


,example_id,gold_label,predicted_label,llm_predicted_label
0,4,Complementary information,Complementary information,Complementary information
1,7,Conflict due to misinformation,No conflict,No conflict
2,12,Conflict due to outdated information,Conflict due to outdated information,No conflict
3,19,Complementary information,Complementary information,Conflict due to outdated information
4,39,No conflict,No conflict,No conflict
...,...,...,...,...
67,421,No conflict,No conflict,No conflict
68,424,No conflict,No conflict,Conflict due to misinformation
69,426,Complementary information,Complementary information,Complementary information
70,435,Conflict due to outdated information,No conflict,No conflict



LLM Pipeline generator preview: Complementary information 
 Answer the following question based on the retrieved search results.
Cite the relevant result number (for example [1]) for each factual sentence.
Provide a well-supported answer to the question. Do not invent unsupported facts.

Question:
when was coffee first made into a drink?

Search results:
[Document 1]
Title: History of coffee - Wikipedia
Source date: 2025-03-31
Relevant passage: In 1580 the Venetian botanist and physician Prospero Alpini imported coffee into the Republic of Venice from Egypt,[18]: 610? and soon coffee shops started opening one by one when coffee spread and became the drink of the intellectuals, of social gatherings, even of lovers as plates of chocolate and coffee were considered a romantic gift. By the year 1763 Venice alone accounted for more than 200 coffee shops,[54] and the health benefits of the miraculous drink were celebrated by many. Some representatives of the Catholic Church opposed coffee a

## 13. Generate and save four conditions

Use a persistent answer cache **inside the generation results JSONL**. Identical complete generator requests (`prompt`, `model`, `temperature`, `max_output_tokens`) reuse one answer, even across methods or after restarting. The saved row retains its own condition and routed label. API-billed token counts are recorded separately from the cached answer's token counts.

Do not mix files from different experiment configurations. The output JSONL is append-only and can be resumed safely.


In [17]:
experiment_config = {
    "dataset_sha256": dataset_sha256,
    "split_sha256": hashlib.sha256(SPLIT_PATH.read_bytes()).hexdigest(),
    "checkpoint_sha256": checkpoint_sha256,
    "selected_seed": selected_seed,
    "selected_val_selection_f1_4": selected_val_f1_4,
    "seed_selection_sha256": selection_sha256,
    "selection_is_provisional": selection_is_provisional,
    "run_mode": RUN_MODE,
    "generator_model": GENERATION_MODEL,
    "llm_classifier_model": LLM_CLASSIFIER_MODEL,
    "llm_classifier_config_hash": llm_classifier_config_hash,
    "temperature": TEMPERATURE,
    "max_output_tokens": MAX_OUTPUT_TOKENS,
    "router_version": ROUTER_VERSION,
    "prompt_version": PROMPT_VERSION,
    "conflict_taxonomy": CONFLICT_TAXONOMY,
    "standard_instruction": STANDARD_INSTRUCTION,
    "shared_prompt_prefix": SHARED_PROMPT_PREFIX,
    "evaluation_protocol_version": "fourway_conflict_aware_rag",
    "document_fields": ["title", "date", "short_text"],
    "date_policy": "original source date when nonempty and not NA; otherwise omitted",
    "original_data_sha256": ORIGINAL_DATA_SHA256,
    "conditions": CONDITIONS,
    "citation_requirement": "inline_citation_per_factual_sentence",
    "classification_evidence_fields": ["question", "title", "short_text"],
}
config_hash = hashlib.sha256(json.dumps(
    experiment_config, ensure_ascii=False, sort_keys=True
).encode()).hexdigest()
print("Experiment fingerprint:", config_hash[:16])


def generation_request_hash(prompt):
    """Hash the full API request, not the classifier name or conflict label."""
    request = {
        "prompt": prompt,
        "model": GENERATION_MODEL,
        "temperature": TEMPERATURE,
        "max_output_tokens": MAX_OUTPUT_TOKENS,
    }
    return hashlib.sha256(json.dumps(
        request, ensure_ascii=False, sort_keys=True
    ).encode()).hexdigest()


def append_jsonl(path, record):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()


def read_generation_cache(path):
    """Recover completed conditions and reusable answers from the saved rows."""
    saved = load_jsonl(path) if Path(path).is_file() else []
    completed = set()
    answers_by_request = {}
    expected_pairs = {(int(i), c) for i in run_df.example_id for c in CONDITIONS}
    for record in saved:
        if record.get("config_hash") != config_hash:
            raise ValueError(f"Different experiment configuration in {path}; archive that file first")
        pair = (int(record["example_id"]), record["condition"])
        if pair in completed or pair not in expected_pairs:
            raise ValueError(f"Duplicate or unexpected saved pair: {pair}")
        completed.add(pair)
        request_hash = generation_request_hash(record["prompt"])
        if record.get("generation_request_hash") != request_hash:
            raise ValueError(f"Missing or incorrect generation request fingerprint: {pair}")
        result = {field: record.get(field) for field in (
            "answer", "input_tokens", "output_tokens", "total_tokens", "near_token_limit"
        )}
        if not isinstance(result["answer"], str) or not result["answer"].strip():
            raise ValueError(f"Empty cached answer: {pair}")
        if request_hash in answers_by_request and answers_by_request[request_hash]["answer"] != result["answer"]:
            raise ValueError(f"Same generator request yielded different saved answers: {pair}")
        answers_by_request.setdefault(request_hash, result)
    return completed, answers_by_request


completed, answers_by_request = read_generation_cache(RESULTS_PATH)
print("Saved conditions:", len(completed), "/", len(run_df) * len(CONDITIONS))
print("Unique generation requests:", len(answers_by_request))

if RUN_API:
    if len(llm_label_map) != len(run_df):
        raise RuntimeError("Complete LLM classification before generating answers")
    for _, row in run_df.iterrows():
        for condition in CONDITIONS:
            pair = (int(row["example_id"]), condition)
            if pair in completed:
                continue
            prompt, routed_label, strategy = build_rag_prompt(row, condition)
            request_hash = generation_request_hash(prompt)
            cache_hit = request_hash in answers_by_request
            if cache_hit:
                result = answers_by_request[request_hash]
            else:
                result = generate_answer(prompt)
                answers_by_request[request_hash] = result
            record = {
                "example_id": int(row["example_id"]),
                "question": row["question"],
                "gold_label": row["gold_label"],
                "predicted_label": row["predicted_label"],
                "llm_predicted_label": row["llm_predicted_label"],
                "condition": condition,
                "routed_label": routed_label,
                "strategy_instruction": strategy,
                "prompt": prompt,
                "answer": result["answer"],
                "generation_request_hash": request_hash,
                "generation_cache_hit": cache_hit,
                "api_input_tokens": 0 if cache_hit else result["input_tokens"],
                "api_output_tokens": 0 if cache_hit else result["output_tokens"],
                "api_total_tokens": 0 if cache_hit else result["total_tokens"],
                "generator_model": GENERATION_MODEL,
                "llm_classifier_model": LLM_CLASSIFIER_MODEL,
                "llm_classifier_config_hash": llm_classifier_config_hash,
                "temperature": TEMPERATURE,
                "max_output_tokens": MAX_OUTPUT_TOKENS,
                "dataset_version": DATASET_VERSION,
                "original_data_sha256": ORIGINAL_DATA_SHA256,
                "classifier_version": CLASSIFIER_VERSION,
                "bert_checkpoint": checkpoint_path.name,
                "bert_checkpoint_sha256": checkpoint_sha256,
                "selected_seed": selected_seed,
                "selected_val_selection_f1_4": selected_val_f1_4,
                "seed_selection_sha256": selection_sha256,
                "selection_is_provisional": selection_is_provisional,
                "router_version": ROUTER_VERSION,
                "prompt_version": PROMPT_VERSION,
                "config_hash": config_hash,
                "evaluation_protocol_version": "fourway_conflict_aware_rag",
                "run_mode": RUN_MODE,
                "input_tokens": result["input_tokens"],
                "output_tokens": result["output_tokens"],
                "total_tokens": result["total_tokens"],
                "near_token_limit": result["near_token_limit"],
            }
            append_jsonl(RESULTS_PATH, record)
            completed.add(pair)
            print("Saved:", pair, "|", len(completed), "/", len(run_df) * len(CONDITIONS),
                  "| cache hit:", cache_hit, flush=True)
else:
    print("RUN_API=False: existing files inspected; no API calls made")


Experiment fingerprint: c6cab3e7dafbda3b
Saved conditions: 0 / 288
Unique generation requests: 0
Saved: (4, 'standard') | 1 / 288 | cache hit: False
Saved: (4, 'llm_pipeline') | 2 / 288 | cache hit: False
Saved: (4, 'bert') | 3 / 288 | cache hit: True
Saved: (4, 'oracle') | 4 / 288 | cache hit: True
Saved: (7, 'standard') | 5 / 288 | cache hit: False
Saved: (7, 'llm_pipeline') | 6 / 288 | cache hit: False
Saved: (7, 'bert') | 7 / 288 | cache hit: True
Saved: (7, 'oracle') | 8 / 288 | cache hit: False
Saved: (12, 'standard') | 9 / 288 | cache hit: False
Saved: (12, 'llm_pipeline') | 10 / 288 | cache hit: False
Saved: (12, 'bert') | 11 / 288 | cache hit: False
Saved: (12, 'oracle') | 12 / 288 | cache hit: True
Saved: (19, 'standard') | 13 / 288 | cache hit: False
Saved: (19, 'llm_pipeline') | 14 / 288 | cache hit: False
Saved: (19, 'bert') | 15 / 288 | cache hit: False
Saved: (19, 'oracle') | 16 / 288 | cache hit: True
Saved: (39, 'standard') | 17 / 288 | cache hit: False
Saved: (39, 'll

## 14. Inspect generation results

Review coverage, identical-request consistency, actual billed tokens and answers approaching the output-token limit.


In [18]:
if Path(RESULTS_PATH).exists():

    results = load_jsonl(
        RESULTS_PATH
    )

    results_df = pd.DataFrame(
        results
    )

    print("Saved outputs:", len(results_df))

    display(
        results_df[
            [
                "example_id",
                "condition",
                "gold_label",
                "predicted_label",
                "llm_predicted_label",
                "answer",
                "output_tokens",
                "near_token_limit",
                "generation_cache_hit",
            ]
        ]
    )


    print("\nOutputs per condition:")

    display(
        results_df[
            "condition"
        ].value_counts()
    )


    print("\nActual generator API tokens by condition (cache hits = 0):")

    display(
        results_df
        .groupby("condition")[
            [
                "api_input_tokens",
                "api_output_tokens",
                "api_total_tokens",
            ]
        ]
        .sum()
    )


    near_limit = results_df[
        results_df["near_token_limit"] == True
    ]

    print(
        "\nAnswers near the output-token limit:",
        len(near_limit),
    )

    if len(near_limit) > 0:
        display(
            near_limit[
                [
                    "example_id",
                    "condition",
                    "output_tokens",
                    "answer",
                ]
            ]
        )

else:
    print(
        "No output file yet:",
        RESULTS_PATH,
    )


Saved outputs: 288


,example_id,condition,gold_label,predicted_label,llm_predicted_label,answer,output_tokens,near_token_limit,generation_cache_hit
0,4,standard,Complementary information,Complementary information,Complementary information,Coffee was first developed into a hot drink du...,55,False,False
1,4,llm_pipeline,Complementary information,Complementary information,Complementary information,Coffee was first made into a drink in the 11th...,57,False,False
2,4,bert,Complementary information,Complementary information,Complementary information,Coffee was first made into a drink in the 11th...,57,False,True
3,4,oracle,Complementary information,Complementary information,Complementary information,Coffee was first made into a drink in the 11th...,57,False,True
4,7,standard,Conflict due to misinformation,No conflict,No conflict,"The expression ""a bridge too far"" originated f...",148,False,False
...,...,...,...,...,...,...,...,...,...
283,435,oracle,Conflict due to outdated information,No conflict,No conflict,"As of the latest information available, only o...",37,False,False
284,452,standard,Conflict due to outdated information,No conflict,No conflict,"No, Tesla and X Corp. do not have the same CEO...",54,False,False
285,452,llm_pipeline,Conflict due to outdated information,No conflict,No conflict,"No, Tesla and X Corp. do not have the same CEO...",41,False,False
286,452,bert,Conflict due to outdated information,No conflict,No conflict,"No, Tesla and X Corp. do not have the same CEO...",41,False,True



Outputs per condition:


condition
standard        72
llm_pipeline    72
bert            72
oracle          72
Name: count, dtype: int64


Actual generator API tokens by condition (cache hits = 0):


,api_input_tokens,api_output_tokens,api_total_tokens
condition,,,
bert,117197,4386,121583
llm_pipeline,229457,7984,237441
oracle,42758,1533,44291
standard,225097,7890,232987



Answers near the output-token limit: 0


In [19]:
if Path(RESULTS_PATH).is_file():
    comparison = results_df.pivot(
        index=["example_id", "question", "gold_label", "predicted_label", "llm_predicted_label"],
        columns="condition", values="answer",
    ).reset_index()
    display(comparison)


condition,example_id,question,gold_label,predicted_label,llm_predicted_label,bert,llm_pipeline,oracle,standard
0,4,when was coffee first made into a drink?,Complementary information,Complementary information,Complementary information,Coffee was first made into a drink in the 11th...,Coffee was first made into a drink in the 11th...,Coffee was first made into a drink in the 11th...,Coffee was first developed into a hot drink du...
1,7,Where did the expression a bridge too far come...,Conflict due to misinformation,No conflict,No conflict,"The expression ""a bridge too far"" originated f...","The expression ""a bridge too far"" originated f...","The expression ""a bridge too far"" originated f...","The expression ""a bridge too far"" originated f..."
2,12,When does this year's Passover start?,Conflict due to outdated information,Conflict due to outdated information,No conflict,This year's Passover begins at sundown on Mond...,This year's Passover begins at sundown on Mond...,This year's Passover begins at sundown on Mond...,This year's Passover begins at sundown on Mond...
3,19,when was the last person executed,Complementary information,Complementary information,Conflict due to outdated information,The last person executed was Alan Eugene Mille...,The last person executed was Alan Eugene Mille...,The last person executed was Alan Eugene Mille...,The last person executed was Alan Eugene Mille...
4,39,How many times csk reached final in ipl?,No conflict,No conflict,No conflict,Chennai Super Kings (CSK) have reached the IPL...,Chennai Super Kings (CSK) have reached the IPL...,Chennai Super Kings (CSK) have reached the IPL...,Chennai Super Kings (CSK) have reached the IPL...
...,...,...,...,...,...,...,...,...,...
67,421,who wrote the song photograph by ringo starr?,No conflict,No conflict,No conflict,"The song ""Photograph"" by Ringo Starr was co-wr...","The song ""Photograph"" by Ringo Starr was co-wr...","The song ""Photograph"" by Ringo Starr was co-wr...","The song ""Photograph"" by Ringo Starr was co-wr..."
68,424,who was the first elected president of pakistan?,No conflict,No conflict,Conflict due to misinformation,The first elected President of Pakistan was Ma...,The first elected President of Pakistan was Ma...,The first elected President of Pakistan was Ma...,The first elected President of Pakistan was Ma...
69,426,what is the meaning of auv in cars?,Complementary information,Complementary information,Complementary information,"In the context of cars, ""AUV"" stands for ""Asia...","In the context of cars, ""AUV"" stands for ""Asia...","In the context of cars, ""AUV"" stands for ""Asia...","In the context of cars, ""AUV"" stands for ""Asia..."
70,435,How many publicly traded American companies ha...,Conflict due to outdated information,No conflict,No conflict,Only one publicly traded American company has ...,Only one publicly traded American company has ...,"As of the latest information available, only o...",Only one publicly traded American company has ...


## 15. Run sequence

1. Place the frozen dataset and checkpoint in the workspace. Set `RUN_API=False` and inspect the prompt previews.
2. For exploratory Validation, the provisional checkpoint can be used with `validation_smoke` or `validation_full`.
3. When ready, set `RUN_API=True`. Classification and generation append their results and resume from consistent JSONL files.
4. Before final Test, fill `BERT_SEED_CANDIDATES` with real Validation scores, confirm the selected checkpoint, freeze the router and generation prompt, then set both confirmation flags.
5. Run `final_test` once with the frozen protocol. Use Stage 3 with the matching evaluation mode and generation filename.

**Important:** old versioned outputs are never automatically migrated. Preserve them as separate experiment records.


## 16. Interpretation limits

- Both classifiers see `question + title + short_text`, but tokenization and context limits differ.
- Oracle receives the expert label by design. BERT and prompted LLM do not receive gold labels.
- Type-conditioned generation is a deterministic mapping from predicted label to instruction, not an autonomous agent.
- Sharing identical generation requests removes avoidable rerun differences; it does not make distinct prompts deterministic.
- BERT checkpoint selection uses Validation metrics rather than Test or RAG generation scores.


## 17. Verify final Test coverage


In [20]:
# Run after final generation. No API requests are made here.
if Path(FINAL_TEST_RESULTS_PATH).exists():
    final_records = load_jsonl(FINAL_TEST_RESULTS_PATH)
    final_outputs = pd.DataFrame(final_records)
    expected_pairs = {(int(i), condition) for i in test_indices for condition in CONDITIONS}
    actual_pairs = [(int(r.example_id), r.condition) for r in final_outputs.itertuples(index=False)]
    assert len(actual_pairs) == len(set(actual_pairs)), "Duplicate example–condition pairs"
    assert set(actual_pairs) == expected_pairs, "Missing or unexpected example–condition pairs"
    assert len(final_outputs) == 72 * 4, "Expected 288 test answers"
    assert final_outputs["answer"].fillna("").astype(str).str.strip().ne("").all()
    assert final_outputs["config_hash"].nunique() == 1
    assert final_outputs["run_mode"].eq("final_test").all()
    assert final_outputs["config_hash"].iloc[0] == config_hash, "Saved outputs from another configuration"
    print("Verified: 72 test questions × 4 conditions = 288 answers")
    display(final_outputs["condition"].value_counts().sort_index())
    if "near_token_limit" in final_outputs:
        print("Answers near token limit:", int(final_outputs["near_token_limit"].fillna(False).sum()))
    final_outputs[["example_id", "condition", "gold_label", "predicted_label", "llm_predicted_label", "answer"]].to_csv(
        OUTPUT_DIR / "rag_generation_index.csv", index=False
    )
else:
    print("No final-test output yet:", FINAL_TEST_RESULTS_PATH)

Verified: 72 test questions × 4 conditions = 288 answers


condition
bert            72
llm_pipeline    72
oracle          72
standard        72
Name: count, dtype: int64

Answers near token limit: 0
